#SS3 Part B

Dataset Loading

The dataset has been loaded into the object "stadoequities". I have inspected the different characteristics of the dataset and found that it holds 41910 instances with 10 different features. The target variable, which is "is_fraud", only contains 2 classes. Since fraud data is highly skewed the dataset is imbalanced. The two classes "0" and "1" represent "No" and "Yes" respectively. I have observed that 98% of the observations are "0" for no fraud.

In [1]:
#installing required libraries
import pandas as pd
import numpy as np

In [2]:
#Loading the dataset
stadioequities = pd.read_csv('../Datasets/stadioequities_transactions.csv')


In [3]:
#Inspecting the dataset
stadioequities.head()

,transaction_id,client_id,timestamp,amount,payment_method,device_type,device_number,ip_address,city,is_fraud
0,t-032058,c-00100,2026-04-17 08:32:02,951.01,CREDIT,desktop,3,102.19.40.20,Nelspruit,0
1,t-014612,c-00932,2025-04-07 15:47:05,7961.21,PAYSHAP,cellphone,4,196.5.77.188,Bloemfontein,0
2,t-013684,c-04217,2025-08-17 08:34:21,5596.87,CARD,cellphone,2,41.64.183.7,Johannesburg,0
3,t-023481,c-04288,2026-07-27 14:29:37,20362.18,DEBIT,cellphone,2,155.203.13.103,Cape Town,0
4,t-014944,c-03762,2024-10-28 09:21:53,6395.11,EFT,cellphone,4,41.122.84.222,Pretoria,0


In [4]:
#inspecting the shape of the dataset
stadioequities.shape

(41910, 10)

In [5]:
stadioequities.info()

<class 'pandas.DataFrame'>
RangeIndex: 41910 entries, 0 to 41909
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  41910 non-null  str    
 1   client_id       41910 non-null  str    
 2   timestamp       41910 non-null  str    
 3   amount          41910 non-null  float64
 4   payment_method  41910 non-null  str    
 5   device_type     41910 non-null  str    
 6   device_number   41910 non-null  int64  
 7   ip_address      41415 non-null  str    
 8   city            41910 non-null  str    
 9   is_fraud        41910 non-null  int64  
dtypes: float64(1), int64(2), str(7)
memory usage: 3.2 MB


In [6]:
#Inspecting class imbalance
stadioequities['is_fraud'].value_counts()

is_fraud
0    41318
1      592
Name: count, dtype: int64

In [7]:
#Imbalance proportions
stadioequities['is_fraud'].value_counts(normalize=True)

is_fraud
0    0.985874
1    0.014126
Name: proportion, dtype: float64

Dataset Preprocessing

In the following Section I have focused on Data preprocessing, whereby the following matters are addressed:

1. Feature Engineering

The Majority of the features were created using the original dataset prior to splitting the features added were 'time_since_last_transaction', 'tx_count_prior_24h', 'is_new_city', 'is_new_device','ip_network_freq'. I added these features so that the model could have more context about the actual transactions that were occuring. After adding the features, I removed the features which do not add to the model training such as the unique identifiers.

2. Spliting the Data

The target variable was split from the rest of the data set. The dataset was then split into the training and test set so that we can effectively train and then test the model.

3. Missing Values

I checked for missing values and found that my final training and testing datasets had no missing values present.

4. Encoding

The dataset was encoded with One Hot Encoding to ensure that the model could effectively make use of the categorical features. 

5. Scale Features

The features were scaled to ensure that the smaller values were not dominated by the bigger values within the dataset.

6. class imbalance - SMOTE

The dataset is highly imbalanced and SMOTE was used to address that class imbalance in the target variable. SMOTE was applied to the training set to ensure that the dataset is balanced. I am using SMOTE because it has previously yielded good results in balancing datasets.

In [8]:
#importing the required libraries
from sklearn.model_selection import train_test_split

In [9]:
# Feauture engineering prior to splitting the dataset into features and target variable

# Sorting Client data 
stadioequities = stadioequities.sort_values(by=['client_id', 'timestamp']).reset_index(drop=True)

In [10]:
# Converting the timestamp column to datetime format
stadioequities['timestamp'] = pd.to_datetime(stadioequities['timestamp'], format = 'mixed')

In [11]:
#Adding the feature - time since last transaction for each client
stadioequities['time_since_last_transaction'] = stadioequities.groupby('client_id')['timestamp'].diff().dt.total_seconds()
stadioequities['time_since_last_transaction'] = stadioequities['time_since_last_transaction'].fillna(-1)

In [12]:
stadioequities = stadioequities.sort_values('timestamp')
stadioequities = stadioequities.set_index('timestamp')
stadioequities['tx_count_prior_24h'] = stadioequities.groupby('client_id')['amount'].transform(
    lambda x: x.rolling('24h').count() - 1
)
stadioequities = stadioequities.reset_index()

# I have ensured that the columns are in the correct order for the next steps

In [13]:
#Adding the feature - is_new_city each transaction
#This feature indicates whether the transaction originates from a new city for the client. a transaction from a new city is an indicator of potential fraud.
stadioequities['is_new_city'] = (~stadioequities.duplicated(subset=['client_id', 'city'])).astype(int)

In [14]:
#Adding the feature - is_new_device for each transaction
#This feature indicates whether the transaction originates from a new device for the client. a transaction from a new device is an indicator of potential fraud.
stadioequities['device_combo'] = stadioequities['device_type'].astype(str) + "_" + stadioequities['device_number'].astype(str)
stadioequities['is_new_device'] = (~stadioequities.duplicated(subset=['client_id', 'device_combo'])).astype(int)

In [15]:
# Adding the feature - IP Network Prefix: Extract the first two octets (e.g., '192.168' from '192.168.1.5')
# This feature can help identify if transactions are coming from the same network, which may be relevant for fraud detection.
stadioequities['ip_network_prefix'] = stadioequities['ip_address'].apply(
    lambda x: '.'.join(str(x).split('.')[:2]) if pd.notna(x) and str(x) != 'nan' else 'unknown'
)

In [16]:
#Dropping the columns that are not needed for the model training
cols_to_drop = [
    'transaction_id', 
    'client_id', 
    'ip_address', 
    'device_number', 
    'device_combo'
]
stadioequities = stadioequities.drop(columns=cols_to_drop)

I have effectively added the new features I can now proceed in splitting and cleaning the data 

In [17]:
#Splitting the dataset into features and target variable
X = stadioequities.drop(columns = ['is_fraud'])
y = stadioequities['is_fraud']

In [18]:
#Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

In [19]:
#checking for missing values in the training set
X_train.isnull().sum()

timestamp                      0
amount                         0
payment_method                 0
device_type                    0
city                           0
time_since_last_transaction    0
tx_count_prior_24h             0
is_new_city                    0
is_new_device                  0
ip_network_prefix              0
dtype: int64

In [20]:
#checking for missing values in the target variable
y_train.isnull().sum()


np.int64(0)

I have confirmed that there is no missing data.

In [21]:
X_train

,timestamp,amount,payment_method,device_type,city,time_since_last_transaction,tx_count_prior_24h,is_new_city,is_new_device,ip_network_prefix
16222,2025-06-10 16:29:50,1061.40,DEBIT,cellphone,Cape Town,875004.0,0.0,0,1,102.182
36268,2026-05-23 23:47:02,1077.84,CREDIT,laptop,Kimberley,5621287.0,0.0,0,0,196.52
5630,2024-12-10 10:59:00,912.33,DEBIT,laptop,Cape Town,-1.0,0.0,1,1,196.250
36648,2026-05-31 10:10:05,22213.99,DEBIT,cellphone,Polokwane,7237051.0,0.0,0,0,41.93
14980,2025-05-19 14:30:35,494.75,DEBIT,desktop,Pretoria,1891065.0,0.0,0,0,105.147
...,...,...,...,...,...,...,...,...,...,...
27535,2025-12-22 13:46:15,299.42,EFT,cellphone,Durban,33160825.0,0.0,0,0,196.197
7797,2025-01-15 12:25:01,2571.73,CREDIT,laptop,Gqeberha,3092106.0,0.0,0,1,102.110
16526,2025-06-15 19:02:15,411.39,EFT,desktop,Centurion,7468347.0,0.0,1,0,41.35
10125,2025-02-24 20:58:00,697.77,EFT,cellphone,Durban,533952.0,0.0,0,0,105.33


In [22]:
#checking the data type of the training set
X_train.dtypes

timestamp                      datetime64[us]
amount                                float64
payment_method                            str
device_type                               str
city                                      str
time_since_last_transaction           float64
tx_count_prior_24h                    float64
is_new_city                             int64
is_new_device                           int64
ip_network_prefix                         str
dtype: object

I want to add another feature which measures the frequency of transactions being performed per IP network(through checking the prefix)

In [23]:
#Adding the feature - ip_network_freq
# This feature can help map the frequency of transactions coming from the same network, which may be relevant for fraud detection.

#Calculating the frequency of each IP network prefix in the training set
ip_frequencies = X_train['ip_network_prefix'].value_counts(normalize=True)

#Mapping the frequencies to the training and testing sets
X_train['ip_network_freq'] = X_train['ip_network_prefix'].map(ip_frequencies)
X_test['ip_network_freq'] = X_test['ip_network_prefix'].map(ip_frequencies).fillna(0)

#dropping the original 'ip_network_prefix' column as we now have the frequency feature
X_train = X_train.drop(columns=['ip_network_prefix'])
X_test = X_test.drop(columns=['ip_network_prefix'])

In [24]:
#dropping the original 'timestamp' column as our model cannot handle datetime features directly
X_train = X_train.drop(columns=['timestamp'])
X_test = X_test.drop(columns=['timestamp'])

In [25]:
X_train

,amount,payment_method,device_type,city,time_since_last_transaction,tx_count_prior_24h,is_new_city,is_new_device,ip_network_freq
16222,1061.40,DEBIT,cellphone,Cape Town,875004.0,0.0,0,1,0.000775
36268,1077.84,CREDIT,laptop,Kimberley,5621287.0,0.0,0,0,0.000686
5630,912.33,DEBIT,laptop,Cape Town,-1.0,0.0,1,1,0.000537
36648,22213.99,DEBIT,cellphone,Polokwane,7237051.0,0.0,0,0,0.001253
14980,494.75,DEBIT,desktop,Pretoria,1891065.0,0.0,0,0,0.001223
...,...,...,...,...,...,...,...,...,...
27535,299.42,EFT,cellphone,Durban,33160825.0,0.0,0,0,0.000775
7797,2571.73,CREDIT,laptop,Gqeberha,3092106.0,0.0,0,1,0.000746
16526,411.39,EFT,desktop,Centurion,7468347.0,0.0,1,0,0.001014
10125,697.77,EFT,cellphone,Durban,533952.0,0.0,0,0,0.000895


In [26]:
# Encoding the datasets
from sklearn.preprocessing import OneHotEncoder

categorical_features = ['payment_method', 'city', 'device_type']

# Standardize text to uppercase to prevent duplicate columns
for col in categorical_features:
    X_train[col] = X_train[col].astype(str).str.upper().str.strip()
    X_test[col] = X_test[col].astype(str).str.upper().str.strip()

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

#fitting the encoder on the training set and transforming both sets
X_train_encoded = encoder.fit_transform(X_train[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

# Creating DataFrames from the encoded features
encoded_columns = encoder.get_feature_names_out(categorical_features)

X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=encoded_columns, index=X_train.index)
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=encoded_columns, index=X_test.index)

# Dropping the original categorical columns from the training and testing sets
X_train = X_train.drop(columns=categorical_features)
X_test = X_test.drop(columns=categorical_features)

# Concatenating the encoded features back to the training and testing sets
X_train = pd.concat([X_train, X_train_encoded_df], axis=1)
X_test = pd.concat([X_test, X_test_encoded_df], axis=1)


In [27]:
#testing output
X_train

,amount,time_since_last_transaction,tx_count_prior_24h,is_new_city,is_new_device,ip_network_freq,payment_method_CARD,payment_method_CREDIT,payment_method_DEBIT,payment_method_EFT,...,city_PIETERMARITZBURG,city_POLOKWANE,city_PRETORIA,city_RUSTENBURG,city_SOWETO,device_type_CELLPHONE,device_type_DESKTOP,device_type_LAPTOP,device_type_SMART_TV,device_type_TABLET
16222,1061.40,875004.0,0.0,0,1,0.000775,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
36268,1077.84,5621287.0,0.0,0,0,0.000686,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5630,912.33,-1.0,0.0,1,1,0.000537,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
36648,22213.99,7237051.0,0.0,0,0,0.001253,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
14980,494.75,1891065.0,0.0,0,0,0.001223,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27535,299.42,33160825.0,0.0,0,0,0.000775,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
7797,2571.73,3092106.0,0.0,0,1,0.000746,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
16526,411.39,7468347.0,0.0,1,0,0.001014,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
10125,697.77,533952.0,0.0,0,0,0.000895,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


The data is ready to be scaled and balanced using SMOTE

In [28]:
# Importing the required library for scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

#scale the data using the StandardScaler
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [29]:
# Importing the required SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

# Applying SMOTE to the training set
X_train_final, y_train_final = smote.fit_resample(X_train_scaled, y_train)

In [30]:
#testing the balance of the final training set
y_train_final.value_counts()

is_fraud
0    33054
1    33054
Name: count, dtype: int64

Model Training

Model training will use grid search to optimize hyperparameters

Model is the Random Forest - Based on our experimental work it is the chosen model to solve this problem, as it works the best


In [ ]:
# Importing the required libraries for model training
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

## Define the parameter grid boundaries
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5]
}

# Initialize the base Random Forest model
rf_base = RandomForestClassifier(random_state=42)

# Configure Grid Search
grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=-1,
)

# Training the model using the Grid Search on the SMOTE-balanced data
grid_search.fit(X_train_final, y_train_final)

# Extract the best model for validation
print("Optimal Hyperparameters found by Grid Search:")
print(grid_search.best_params_)

rf_model = grid_search.best_estimator_

Optimal Hyperparameters found by Grid Search:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}


Model Validation

Results Visualisation